# 02 — Model Definition (YOLOv3 from scratch)

Darknet-53 backbone + 3-scale detection head, implemented directly in PyTorch
(`src/model.py`) rather than via a library, so every `Conv2d` is a plain
`nn.Conv2d` the Tucker-2 pipeline can enumerate and replace later — same
convention as the VGG/ResNet/ViT pipeline.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath("../src"))
import torch
from model import YOLOv3, count_conv_params

with open("../checkpoints/run_config.json") as f:
    cfg = json.load(f)
NUM_CLASSES = len(cfg["class_names"])
IMG_SIZE = cfg["img_size"]
print("classes:", cfg["class_names"])

In [ ]:
model = YOLOv3(num_classes=NUM_CLASSES)
total, conv = count_conv_params(model)
print(f"total params: {total:,}")
print(f"conv params:  {conv:,}  ({conv/total:.1%} of total)")
print(f"num Conv2d layers: {sum(1 for m in model.modules() if isinstance(m, torch.nn.Conv2d))}")

## Forward-pass shape check (3 output scales: stride 32 / 16 / 8)

In [ ]:
x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    outs = model(x)
for scale_name, o in zip(["large (stride32)", "medium (stride16)", "small (stride8)"], outs):
    print(f"{scale_name}: {tuple(o.shape)}   -> anchors*(5+{NUM_CLASSES}) = {3*(5+NUM_CLASSES)} channels")

## Save an untrained checkpoint (useful as a reset point for the demo)

In [ ]:
os.makedirs("../checkpoints", exist_ok=True)
torch.save(model.state_dict(), "../checkpoints/yolov3_untrained.pt")
print("saved ../checkpoints/yolov3_untrained.pt")